# 10c Classify Abstract Concepts

Classify each unique abstract against the reviewed catalog using Jev typed NOUL judgments. The classifier receives a detailed, versioned rubric and the available concept metadata, but no pair identifiers or SI outcomes. Results resume from a versioned per-concept cache.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from amr.pipeline_config import cfg, load_pipeline_config
from amr.pipelines.concept_notebook_workflows import load_classification_context, run_cached_concept_classification
from amr.pipeline_config import cfg, load_pipeline_config
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
RUN_JEV_CLASSIFICATION = True
MAX_CALLS = None  # Resume cached work; None completes every missing abstract/concept label.
MAX_WORKERS = int(cfg(PIPELINE_CONFIG, "system1.max_workers", 128))
CONTEXT = load_classification_context(PROJECT_ROOT)
print(f"Catalog {CONTEXT['catalog_version']}; {len(CONTEXT['abstracts']):,} unique abstracts; {len(CONTEXT['catalog'])} concepts")


## Preview prompt without making a request

Review the sample state, global rubric, and full question for the most fully specified concept before enabling calls. The state contains title and abstract text only. The question contains every non-empty concept record field. Jev receives no PMID, pathogen, antibiotic, SI value, split assignment, or future outcome.

In [ ]:
if CONTEXT["preview"]:
    sample = CONTEXT["preview"]
    print(f"State sent to Jev:\nTitle: {sample['title']}\nAbstract: {sample['abstract'][:2000]}")
    print("Prompt version:", CONTEXT["prompt_version"])
    print("Global classification rubric:\n", CONTEXT["classification_rubric"])
    print("Example concept question:\n", CONTEXT["preview_question"])


Jev returns a NOUL score from 0 to 1 for each concept. The score is Jev’s confidence in the abstract-presence judgment under the prompt rubric; it is not calibrated as a probability and does not represent importance, temporal effect, or causal influence. Blank abstracts are skipped. Successful scores are keyed by PMID, concept-definition hash, Jev model, and prompt version. When a catalog is expanded, unchanged concepts are reused; Jev is asked only for missing or revised concept labels under the same prompt version. A prompt-version change intentionally invalidates reuse of old labels so the new rubric is applied consistently.

`MAX_CALLS=None` processes all pending abstract/concept labels in a resumable run. `MAX_WORKERS` uses the configured System 1 worker count. `RUN_JEV_CLASSIFICATION=True` enables the Jev call cell; execution requires configured OpenRouter credentials. Progress checkpoints report elapsed time, ETA for this session, the latest OpenRouter balance, and an estimated ending balance once OpenRouter reports spend. If interrupted, rerun the cell to continue from cached scores.

Jev returns a NOUL score between 0 and 1 for each concept. Treat this as a model-assigned presence score, not a calibrated probability or a measure of importance. Blank abstracts are skipped. Successful scores are keyed by PMID, concept-definition hash, Jev model, and prompt version. When a catalog is expanded, unchanged concepts are reused; Jev is asked only for missing or revised concept labels, and cached values are linked into the active catalog version. Reruns skip labels already completed for the same concept.

`MAX_CALLS=None` processes all pending abstract/concept labels in a resumable run. `MAX_WORKERS` uses the configured System 1 worker count. `RUN_JEV_CLASSIFICATION=True` enables the Jev call cell; execution requires configured OpenRouter credentials. Progress checkpoints report elapsed time, ETA for this session, the latest OpenRouter balance, and an estimated ending balance once OpenRouter reports spend. If interrupted, rerun the cell to continue from cached scores.

In [ ]:
RESULT = run_cached_concept_classification(
    PROJECT_ROOT, max_calls=MAX_CALLS, max_workers=MAX_WORKERS,
    run_classification=RUN_JEV_CLASSIFICATION,
)
display(RESULT["status_summary"])
